# Molecular phase estimation for LiH

Regenerate the molecular model, estimate its energy and compare with full CI. The quantum circuit and phase marginal are computed locally with FlagQuantum.

## Controlled product formulas are declared, not inferred

The visible builder composes the original grouped Strang evolution powers
under phase controls, followed by inverse QFT. FlagQuantum lowers the checked
native helpers. Host code regenerates the active-space LiH Hamiltonian,
normalization, tapering, evolution times and individual repetition counts.
An error-bound estimate is not a measured compilation or execution error.

Inspect all original phase powers and experimental repetition counts before
changing scale. The full named phase histogram is decoded with the declared
signed phase/energy offset. Sampled peaks differ from exact eigenvalues, and
finite QPE precision, product-formula error and coefficient cutoff are separate.
Propagate builder edits to workflow.py for later runs. All original powers,
independent checks and resource limits remain intact.


Use the local FlagQuantum installation.

In [ ]:
from flagquantum import synth
from flagquantum.synth import Output
from flagquantum.synth import QArray
from flagquantum.synth import QBit
from flagquantum.synth import adjoint
from flagquantum.synth import allocate
from flagquantum.synth import control
from flagquantum.synth import hadamard_transform
from flagquantum.synth import inplace_prepare_amplitudes
from flagquantum.synth import qft
from flagquantum.synth import qfunc
from flagquantum.synth import suzuki_trotter
import math

from examples.synthesis.applications.chemistry.qpe_for_molecules.overview.workflow import (
    Model,
    PHASE_BITS,
    POWERS,
    STEPS,
    WIDTH,
    ZERO_STATE,
    commuting_runs,
    observable,
)


def build_preview(
    model: Model, scale: float = 1.0, amplitudes: tuple[float, ...] = ZERO_STATE
) -> synth.Program:
    """Compose separately specialized Strang evolutions under phase controls."""
    if not math.isfinite(scale) or not 0 < scale <= 2:
        raise ValueError("The positive time scale must not exceed two")
    if len(amplitudes) != 1 << WIDTH or not math.isclose(
        math.fsum(value * value for value in amplitudes), 1, rel_tol=0, abs_tol=1e-12
    ):
        raise ValueError("The six-bit state amplitudes must be normalized")
    g0, g1, g2, g3, g4, g5, g6, g7 = map(observable, commuting_runs(model.terms))

    @qfunc
    def power(time: float, n: int, state: QArray[QBit]) -> None:
        suzuki_trotter(g0, time / (2 * n), 1, 1, state)
        for i in range(n):
            suzuki_trotter(g1, time / (2 * n), 1, 1, state)
            suzuki_trotter(g2, time / (2 * n), 1, 1, state)
            suzuki_trotter(g3, time / (2 * n), 1, 1, state)
            suzuki_trotter(g4, time / (2 * n), 1, 1, state)
            suzuki_trotter(g5, time / (2 * n), 1, 1, state)
            suzuki_trotter(g6, time / (2 * n), 1, 1, state)
            suzuki_trotter(g7, time / n, 1, 1, state)
            suzuki_trotter(g6, time / (2 * n), 1, 1, state)
            suzuki_trotter(g5, time / (2 * n), 1, 1, state)
            suzuki_trotter(g4, time / (2 * n), 1, 1, state)
            suzuki_trotter(g3, time / (2 * n), 1, 1, state)
            suzuki_trotter(g2, time / (2 * n), 1, 1, state)
            suzuki_trotter(g1, time / (2 * n), 1, 1, state)
            if i < n - 1:
                suzuki_trotter(g0, time / n, 1, 1, state)
            else:
                suzuki_trotter(g0, time / (2 * n), 1, 1, state)

    @qfunc
    def selected_power(time: float, n: int, bit: QBit, state: QArray[QBit]) -> None:
        control(bit, lambda: power(time, n, state))

    @qfunc
    def main(
        times: list[float],
        steps: list[int],
        amplitudes: list[float],
        state: Output[QArray[QBit]],
        phase: Output[QArray[QBit]],
    ) -> None:
        allocate(WIDTH, state)
        allocate(PHASE_BITS, phase)
        inplace_prepare_amplitudes(amplitudes, 0, state)
        hadamard_transform(phase)
        for i in range(PHASE_BITS):
            selected_power(times[i], steps[i], phase[i], state)
        with adjoint():
            qft(phase)

    return synth.synthesize(
        main,
        classical_inputs={
            "times": [-2 * math.pi * scale * p for p in POWERS],
            "steps": list(STEPS),
            "amplitudes": list(amplitudes),
        },
        max_qubits=13,
        max_workspace=0,
        max_operations=100_000,
        memory_limit_bytes=64 * 1024**2,
    )


import math
from pathlib import Path

from IPython.display import Image, display

import flagquantum as fq


Import the native workflow and its molecular model.

In [ ]:
from examples.synthesis.applications.chemistry.qpe_for_molecules.overview import (
    model,
    workflow,
)
from examples.synthesis.applications.chemistry.qpe_for_molecules.overview.main import (
    print_report,
    save_figures,
)

print(fq.__name__)


Inspect the candidate geometries and select LiH at 1.596 Å.

In [ ]:
print(model.GEOMETRIES)
print("Selected geometry:", model.GEOMETRIES["LiH"])


Recompute electronic integrals and full and active CI.

In [ ]:
lih = model.build_model()
print("RHF energy:", lih.rhf)


Encode the six-qubit symmetry sector containing the HF determinant.

In [ ]:
print("Sector dimension:", len(model.sector_words()))
print("Hartree–Fock occupation:", model.sector_words()[0])


Keep both exact energy references.

In [ ]:
print("Full CI and active CI:", lih.full_ci, lih.active_ci)


Inspect the freshly tapered coefficient spectrum.

In [ ]:
print(
    "Largest and smallest retained numerical magnitudes:",
    lih.coefficient_magnitudes[0],
    lih.coefficient_magnitudes[-1],
)


Apply the 0.03 Ha coefficient cutoff.

In [ ]:
print("Cutoff and retained terms:", model.CUTOFF, len(lih.terms))


Recover the scale and the removed scalar energy.

In [ ]:
print("Normalization and offset:", lih.normalization, lih.offset)


Compute the ordered second-order commutator bound.

In [ ]:
gamma = model.gamma_two(lih.terms)
print("Gamma and target error:", gamma, workflow.EPS)


Use the original seven phase bits.

In [ ]:
print("Powers:", workflow.POWERS)


Compute the adaptive theoretical schedule.

In [ ]:
r0 = math.sqrt(2**5 * gamma / (3 * workflow.EPS)) * (2 * math.pi) ** 1.5
print("Adaptive bound:", tuple(math.ceil(r0 * p**1.5) for p in workflow.POWERS))


Compare with naïve powers of the first approximation.

In [ ]:
print("Naive bound:", tuple(math.ceil(r0 * p) for p in workflow.POWERS))


Use the experimental per-power step counts in the quantum program.

In [ ]:
print("Experimental repetitions:", workflow.STEPS)


Prepare the Hartree–Fock state in the tapered representation.

In [ ]:
print("Tapered Hartree–Fock word:", model.sector_words().index(model.HF_OCCUPATION))


Build the native controlled product formulas and inverse QFT.

In [ ]:
program = build_preview(lih)
print("Compiled phase register:", len(program.registers["phase"]))


Execute locally and draw a seeded finite sample of 1,000 outcomes.

In [ ]:
result = workflow.read_run(lih, program)
print("Total draws:", sum(result.counts))


Display the coefficient spectrum and the energy histogram.

In [ ]:
figures = Path("notebook-figures")
save_figures(result, figures)
display(Image(filename=str(figures / "coefficient-magnitudes.png")))
display(Image(filename=str(figures / "energy-histogram.png")))


Read the most probable energy and the lowest detected peak alongside the full CI reference.

In [ ]:
print_report(result)
